<a href="https://colab.research.google.com/github/YuxinZhang1998/S3-SIF/blob/main/Step_4_SIF_output.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import ee
import geemap
import random
import datetime
import os
import numpy as np
import pandas as pd
from scipy import signal

ee.Authenticate()
ee.Initialize(project='ee-yuxinzhanguv1')
rf_model= ee.Classifier.load('projects/ee-yuxinzhanguv1/assets/RF_SIF_model')

In [ ]:
pixelsize = 300
startDateStr = '2018-08-10'
endDateStr = '2018-08-31'
S3site = ee.ImageCollection('projects/ee-yuxinzhanguv1/assets/S3-4DAYS-HL-FeatureCollection_round1').filterDate(startDateStr, endDateStr)
FVCsites = ee.ImageCollection('projects/ee-yuxinzhanguv1/assets/GPR-S3-4DAYS-HL-FVC').select(['FVC']).filterDate(startDateStr, endDateStr)
LCCsites = ee.ImageCollection('projects/ee-yuxinzhanguv1/assets/GPR-S3-4DAYS-HL-LCC').select(['LCC']).filterDate(startDateStr, endDateStr)
Geosites = ee.ImageCollection('projects/ee-yuxinzhanguv1/assets/Geo-4DAYS-HL-FeatureCollection')#.select(['longitude', 'latitude']).filterDate(startDateStr, endDateStr)
FVCsite_renamed = FVCsites.map(lambda img: img.rename(['FVC_GREEN']))
LCCsite_renamed = LCCsites.map(lambda img: img.rename(['LCC_GREEN']))
# Geosites_renamed = Geosites.map(
#     lambda img: img.rename(['lon', 'lat'])
# )
export_prefix = 'Predicted_HL_'
asset_prefix = 'projects/ee-yuxinzhanguv1/assets/Predicted_HL_Collection_round1_mask/'
#region = ee.Geometry.Rectangle([-10, 36, 4, 44], geodesic=False)

In [ ]:
def process_image(image):
    date = image.date()
    day_filter = ee.Filter.date(date, date.advance(1, 'day'))

    fvc = FVCsite_renamed.filter(day_filter).first()
    lcc = LCCsite_renamed.filter(day_filter).first()
    geo = Geosites.filter(day_filter).first()

    combined = image.addBands(fvc).addBands(lcc).addBands(geo)


    result = combined.classify(rf_model).toFloat()

    return result.set({
        'system:time_start': image.get('system:time_start'),
        'date_str': date.format('YYYYMMdd'),
        'system:index': image.get('system:index')
    })

predicted_site = S3site.map(process_image)

index_list = S3site.aggregate_array('system:index').getInfo()
date_list = predicted_site.aggregate_array('date_str').getInfo()

print(f"{len(index_list)} images detected, task submission initiated...")

for i in range(len(index_list)):
    try:
        idx = index_list[i]
        date_str = date_list[i]

        image_to_export = predicted_site.filter(ee.Filter.eq('system:index', idx)).first()

        export_name = f"{export_prefix}{date_str}"
        asset_path = f"{asset_prefix}{export_name}"

        task = ee.batch.Export.image.toAsset(
            image=image_to_export,
            description=export_name,
            assetId=asset_path,
            scale=pixelsize,
            maxPixels=1e13
        )
        task.start()
        print(f"Task created successfully: {export_name}")

    except Exception as e:
        print(f"Index {i} processing failed: {e}")

print("All tasks have been submitted. Please check the progress in the GEE console.")

检测到 5 幅影像，开始提交任务...
成功创建任务: Predicted_HL_20180813
成功创建任务: Predicted_HL_20180817
成功创建任务: Predicted_HL_20180821
成功创建任务: Predicted_HL_20180825
成功创建任务: Predicted_HL_20180829
所有任务已提交，请前往 GEE 控制台查看进度。
